# Per-Injection Classifier Analysis

**Goal**: For each injection separately, test whether attention patterns at the first reasoning token
generalize *across user-task contexts within the same suite*.

**Framing**: An attacker knows the target environment (banking / slack / travel / workspace)
but not the specific user task. So the relevant generalization is *within-suite cross-context*,
not cross-injection.

**Method**: Leave-one-context-out (LOCO) cross-validation per injection.
Train on all-but-one user-task context, test on the held-out one. Repeat for every context.
This gives one AUROC per (injection, held-out context) cell — directly answers
"does the signal transfer to a new user task for this environment?"

**Data**: `profiling_logs_v7` — 3 injections × up to 4 contexts each.
Switch `LOG_DIR` to `profiling_logs_v8` when that run completes.

In [ ]:
import sys, sqlite3, json, dill, pickle, torch, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

plt.rcParams.update({
    'figure.dpi': 120, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# ── Architecture ─────────────────────────────────────────────────────────────
N_LAYERS          = 24
N_HEADS           = 64
LOCAL_LAYERS      = list(range(0, N_LAYERS, 2))
GLOBAL_LAYERS     = list(range(1, N_LAYERS, 2))
SINK_POSITIONS    = {0}
N_REASONING_STEPS = 10

# ── Span groups (same as classifier_analysis.ipynb) ──────────────────────────
SPAN_GROUPS = {
    'attack_payload':   {'attack_payload'},
    'attack_prefix':    {'attack_prefix'},
    'attack_suffix':    {'attack_suffix'},
    'user_instruction': {'user_instruction'},
    'tool_env_data':    {'tool_env_data'},
    'dev_instructions': {'developer_instructions'},
    'dev_tools':        {'developer_tools'},
    'system_meta':      {'system_meta'},
    'frame_boundary':   {'frame_boundary'},
    'frame_message':    {'frame_message'},
    'frame_role':       {'frame_role'},
    'frame_channel':    {'frame_channel'},
    'frame_constrain':  {'frame_constrain'},
    'frame_chan_name':  {'frame_channel_name'},
    'frame_cons_type':  {'frame_constrain_type'},
    'frame_metadata':   {'frame_metadata'},
}
GROUP_NAMES = list(SPAN_GROUPS.keys()) + ['sink']
N_GROUPS    = len(GROUP_NAMES)               # 17
N_FEATURES  = N_LAYERS * N_HEADS * N_GROUPS  # 26112 per step

# ── Paths ─────────────────────────────────────────────────────────────────────
LOG_DIR     = Path('profiling_logs_v7')   # ← switch to v8 when ready
TENSOR_BASE = Path('.')
CACHE_FILE  = LOG_DIR / 'feature_cache'   # reuse existing cache from classifier_analysis

print(f'Groups: {N_GROUPS}   Features/step: {N_FEATURES}')
print(f'Log dir: {LOG_DIR}')

In [ ]:
# ── Load DB ───────────────────────────────────────────────────────────────────
conn = sqlite3.connect(LOG_DIR / 'experiment_logs.db')
cur  = conn.cursor()

cur.execute("SELECT metadata_json FROM logs WHERE event='profiling_run'")
runs_meta = {json.loads(r[0])['run_id']: json.loads(r[0]) for r in cur.fetchall()}

cur.execute("SELECT metadata_json, object_data FROM logs WHERE event='profiling_capture'")
cap_by_run = {}
for meta_json, obj_data in cur.fetchall():
    rid = json.loads(meta_json)['run_id']
    cap_by_run[rid] = dill.loads(obj_data)

conn.close()
print(f'Runs: {len(runs_meta)}   Captures: {len(cap_by_run)}')

flip_runs = [
    m for m in runs_meta.values()
    if m.get('perturbation_type') == 'flip' and m['run_id'] in cap_by_run
]
print(f'Flip runs with caps: {len(flip_runs)}')
print(f'  success: {sum(1 for r in flip_runs if r.get("success"))}')
print(f'  failure: {sum(1 for r in flip_runs if not r.get("success"))}')

In [ ]:
# ── Data audit ────────────────────────────────────────────────────────────────
audit = defaultdict(lambda: {'success': 0, 'failure': 0})
for m in flip_runs:
    ctx = f"{m.get('suite')}/{m.get('injection_task_id')}/{m.get('user_task_id')}"
    audit[ctx]['success' if m.get('success') else 'failure'] += 1

audit_df = pd.DataFrame([
    {'context': k, 'suite': k.split('/')[0], 'injection': k.split('/')[1],
     'user_task': k.split('/')[2],
     'success': v['success'], 'failure': v['failure'],
     'total': v['success'] + v['failure'],
     'rate': v['success'] / (v['success'] + v['failure'])}
    for k, v in sorted(audit.items())
])
print(audit_df[['context','success','failure','total','rate']].to_string(
    index=False, float_format='{:.2f}'.format))

# Identify injections present in the data
INJECTIONS = sorted(audit_df[['suite','injection']].drop_duplicates()
                    .apply(lambda r: (r['suite'], r['injection']), axis=1))
print(f'\nInjections: {INJECTIONS}')

In [ ]:
# ── Feature extraction (identical to classifier_analysis.ipynb) ───────────────
def _span_mask(spans, seq_len, tag_set):
    m = torch.zeros(seq_len, dtype=torch.bool)
    for s in spans:
        if s['tag'] in tag_set:
            m[s['start']:min(s['end'], seq_len)] = True
    for p in SINK_POSITIONS:
        if p < seq_len:
            m[p] = False
    return m


def _extract_features(m, ci, cap, n_steps, normalize):
    seq_len = cap['seq_len']
    spans   = ci['spans']
    attn    = cap['attention']

    group_idxs = {}
    for grp, tag_set in SPAN_GROUPS.items():
        mask = _span_mask(spans, seq_len, tag_set)
        group_idxs[grp] = mask.nonzero(as_tuple=False).view(-1)
    sink_idx = torch.tensor([p for p in SINK_POSITIONS if p < seq_len], dtype=torch.long)
    group_idxs['sink'] = sink_idx

    feat_mass = np.zeros((n_steps, N_LAYERS, N_HEADS, N_GROUPS), dtype=np.float32)
    feat_ent  = np.zeros((n_steps, N_LAYERS, N_HEADS, N_GROUPS), dtype=np.float32)

    for li in range(N_LAYERS):
        if li not in attn:
            continue
        a            = attn[li].float()
        actual_steps = min(n_steps, a.shape[1])
        klen         = a.shape[-1]
        for step in range(actual_steps):
            a_step = a[:, step, :]
            for gi, grp in enumerate(GROUP_NAMES):
                idxs = group_idxs[grp]
                idxs = idxs[idxs < klen]
                if len(idxs) == 0:
                    continue
                span_a = a_step[:, idxs]
                feat_mass[step, li, :, gi] = span_a.sum(-1).numpy()
                p   = span_a / (span_a.sum(-1, keepdim=True) + 1e-10)
                ent = -(p * (p + 1e-10).log()).sum(-1)
                feat_ent[step, li, :, gi] = ent.numpy()

    if normalize:
        total = feat_mass.sum(axis=-1, keepdims=True)
        feat_mass = feat_mass / np.where(total == 0, 1.0, total)

    return feat_mass.reshape(n_steps, -1), feat_ent.reshape(n_steps, -1)


def _load_and_extract(args, n_steps, normalize):
    m, ci, path = args
    if not path.exists():
        return None
    try:
        cap = torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        return None
    try:
        fm, fe = _extract_features(m, ci, cap, n_steps, normalize)
    except Exception:
        return None
    finally:
        del cap
    return m, fm, fe


def build_feature_matrix(run_meta_list, n_steps=N_REASONING_STEPS, normalize=True,
                          n_workers=4, cache_file=None, force=False, verbose=True):
    cf = Path(str(cache_file) + '.npz') if cache_file else None
    cm = Path(str(cache_file) + '_meta.pkl') if cache_file else None

    if cf and not force and cf.exists() and cm and cm.exists():
        data         = np.load(cf)
        current_rids = {m['run_id'] for m in run_meta_list if m['run_id'] in cap_by_run}
        if set(data['run_ids'].tolist()) == current_rids:
            if verbose:
                print(f'Cache hit → {cf}')
            with open(cm, 'rb') as f:
                meta_out = pickle.load(f)
            return data['X_mass'], data['X_entropy'], data['y'].astype(np.int32), meta_out
        elif verbose:
            print('Cache stale — recomputing')

    tasks = [
        (m, cap_by_run[m['run_id']], TENSOR_BASE / cap_by_run[m['run_id']]['cap_path'])
        for m in run_meta_list if m['run_id'] in cap_by_run
    ]
    N_tasks   = len(tasks)
    X_mass    = np.empty((N_tasks, n_steps, N_FEATURES), dtype=np.float32)
    X_entropy = np.empty((N_tasks, n_steps, N_FEATURES), dtype=np.float32)
    y_arr     = np.empty(N_tasks, dtype=np.int32)
    meta_out  = [None] * N_tasks
    write_idx = 0
    failed    = 0

    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = [ex.submit(_load_and_extract, t, n_steps, normalize) for t in tasks]
        for fut in tqdm(as_completed(futs), total=N_tasks,
                        desc='Loading', disable=not verbose):
            r = fut.result()
            if r is None:
                failed += 1
                continue
            m, fm, fe = r
            X_mass[write_idx]    = fm
            X_entropy[write_idx] = fe
            y_arr[write_idx]     = int(bool(m.get('success')))
            meta_out[write_idx]  = m
            write_idx += 1

    X_mass, X_entropy = X_mass[:write_idx], X_entropy[:write_idx]
    y_arr   = y_arr[:write_idx]
    meta_out = meta_out[:write_idx]
    run_ids  = np.array([m['run_id'] for m in meta_out])

    if verbose:
        print(f'Processed {write_idx}/{N_tasks} ({failed} failed)')

    if cf:
        np.savez_compressed(cf, X_mass=X_mass, X_entropy=X_entropy,
                            y=y_arr, run_ids=run_ids)
        with open(cm, 'wb') as f:
            pickle.dump(meta_out, f)
        if verbose:
            print(f'Cache saved → {cf}')

    return X_mass, X_entropy, y_arr, meta_out

print('Feature functions defined.')

In [ ]:
# ── Load / build feature matrix ───────────────────────────────────────────────
X_mass_all, X_entropy_all, y, meta = build_feature_matrix(
    flip_runs, n_steps=N_REASONING_STEPS, normalize=True,
    n_workers=4, cache_file=CACHE_FILE,
)

print(f'X_mass_all:    {X_mass_all.shape}')
print(f'y: {y.shape}  success={y.sum()}  failure={(1-y).sum()}')

# Step-0 slices (most informative single step)
X_mass0    = X_mass_all[:, 0, :]     # [N, N_FEATURES]
X_entropy0 = X_entropy_all[:, 0, :]
X_global0  = X_mass_all[:, 0, :].reshape(len(meta), N_LAYERS, N_HEADS, N_GROUPS)\
               [:, GLOBAL_LAYERS, :, :].reshape(len(meta), -1)

# Context keys for masking
ctx_keys = np.array([
    f"{m.get('suite')}/{m.get('injection_task_id')}/{m.get('user_task_id')}"
    for m in meta
])
inj_keys = np.array([
    f"{m.get('suite')}/{m.get('injection_task_id')}"
    for m in meta
])

print('\nPer-injection counts:')
for suite, inj in INJECTIONS:
    mask = inj_keys == f'{suite}/{inj}'
    n_ctx = len(set(ctx_keys[mask]))
    print(f'  {suite}/{inj}: n={mask.sum()}  contexts={n_ctx}  '
          f'success={y[mask].sum()}  failure={(1-y[mask]).sum()}')

In [ ]:
# ── Leave-one-context-out (LOCO) evaluation ───────────────────────────────────

def make_clf():
    return LogisticRegression(
        C=0.01, class_weight='balanced', max_iter=2000,
        solver='saga', random_state=42,
    )


def loco_auroc(X, y, ctx_keys, inj_mask, min_class=5):
    """
    Leave-one-context-out AUROC for one injection.

    For each user-task context c in the injection:
      - train on all other contexts in the same injection
      - test on c
      - compute AUROC

    Returns dict: {context_key: auroc or None}
    """
    X_inj    = X[inj_mask]
    y_inj    = y[inj_mask]
    ctx_inj  = ctx_keys[inj_mask]
    contexts = sorted(set(ctx_inj))

    results = {}
    for held_ctx in contexts:
        test_mask  = ctx_inj == held_ctx
        train_mask = ~test_mask

        y_tr, y_te = y_inj[train_mask], y_inj[test_mask]

        # Skip if held-out context is trivially imbalanced
        if y_te.sum() < min_class or (1 - y_te).sum() < min_class:
            results[held_ctx] = None
            continue
        # Skip if training set is trivially imbalanced
        if y_tr.sum() < min_class or (1 - y_tr).sum() < min_class:
            results[held_ctx] = None
            continue

        clf = make_clf()
        clf.fit(X_inj[train_mask], y_tr)
        prob = clf.predict_proba(X_inj[test_mask])[:, 1]
        results[held_ctx] = roc_auc_score(y_te, prob)

    return results


# Feature sets to compare
FEAT_SETS = {
    'step0_global_mass':         X_global0,
    'step0_mass':                X_mass0,
    'step0_mass+entropy':        np.hstack([X_mass0, X_entropy0]),
}

print('Running LOCO evaluation...')
loco_results = {}   # {feat_name: {inj_key: {ctx: auroc}}}

for feat_name, X_feat in FEAT_SETS.items():
    loco_results[feat_name] = {}
    for suite, inj in INJECTIONS:
        inj_key  = f'{suite}/{inj}'
        inj_mask = inj_keys == inj_key
        res      = loco_auroc(X_feat, y, ctx_keys, inj_mask)
        loco_results[feat_name][inj_key] = res
        valid    = [v for v in res.values() if v is not None]
        mean_auc = np.mean(valid) if valid else float('nan')
        print(f'  {feat_name:35s}  {inj_key:40s}  mean_AUROC={mean_auc:.3f}')

print('Done.')

In [ ]:
# ── LOCO results table ────────────────────────────────────────────────────────
# Best feature set = highest mean AUROC across all injections and contexts.
feat_means = {}
for feat_name in FEAT_SETS:
    vals = [
        v for inj_res in loco_results[feat_name].values()
        for v in inj_res.values() if v is not None
    ]
    feat_means[feat_name] = np.mean(vals) if vals else float('nan')

BEST_FEAT = max(feat_means, key=feat_means.get)
print(f'Feature mean AUROC across all LOCO cells:')
for fn, mv in sorted(feat_means.items(), key=lambda x: -x[1]):
    marker = '  ← best' if fn == BEST_FEAT else ''
    print(f'  {fn:35s}: {mv:.4f}{marker}')

print()
print(f'Best feature set: {BEST_FEAT}')
print()

# Detailed table for best feature set
rows = []
for suite, inj in INJECTIONS:
    inj_key = f'{suite}/{inj}'
    res     = loco_results[BEST_FEAT][inj_key]
    for ctx, auc in sorted(res.items()):
        ut = ctx.split('/')[-1].replace('user_task_', 'ut')
        inj_short = inj.replace('injection_task_', 'it')
        mask  = ctx_keys == ctx
        rows.append({
            'injection':   f'{suite}/{inj_short}',
            'held_out':    ut,
            'n_test':      int(mask.sum()),
            'succ_rate':   float(y[mask].mean()),
            'LOCO_AUROC':  auc if auc is not None else float('nan'),
            'note':        'skipped (imbalanced)' if auc is None else '',
        })

loco_df = pd.DataFrame(rows)
print(loco_df.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# ── LOCO AUROC heatmap: injection × held-out context ─────────────────────────
valid_rows = loco_df[loco_df['LOCO_AUROC'].notna()]
injections = valid_rows['injection'].unique()
all_uts    = sorted(valid_rows['held_out'].unique())

# Build matrix [n_inj, n_ut] — NaN where combination doesn't exist
mat  = np.full((len(injections), len(all_uts)), np.nan)
rate = np.full_like(mat, np.nan)

for ri, inj in enumerate(injections):
    sub = valid_rows[valid_rows['injection'] == inj]
    for row in sub.itertuples():
        if row.held_out in all_uts:
            ci = all_uts.index(row.held_out)
            mat[ri, ci]  = row.LOCO_AUROC
            rate[ri, ci] = row.succ_rate

fig, axes = plt.subplots(1, 2, figsize=(max(10, len(all_uts)*1.4 + 3), len(injections)*1.0 + 2))

for ax, data, title, cmap, vmin, vmax, fmt in [
    (axes[0], mat,  f'LOCO AUROC ({BEST_FEAT})', 'RdYlGn', 0.5, 1.0, '{:.2f}'),
    (axes[1], rate, 'Held-out context success rate', 'Blues',  0.0, 1.0, '{:.2f}'),
]:
    im = ax.imshow(data, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_xticks(range(len(all_uts)))
    ax.set_xticklabels(all_uts, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(injections)))
    ax.set_yticklabels(injections, fontsize=8)
    ax.set_xlabel('Held-out user-task context')
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.7)
    for ri in range(len(injections)):
        for ci in range(len(all_uts)):
            val = data[ri, ci]
            if not np.isnan(val):
                ax.text(ci, ri, fmt.format(val), ha='center', va='center',
                        fontsize=7, color='black')

plt.suptitle('Per-injection LOCO cross-context evaluation\n'
             '(train = all other contexts in same suite, test = held-out)',
             fontsize=10)
plt.tight_layout()
plt.show()

# Summary: mean AUROC per injection
print('\nMean LOCO AUROC per injection (best feat):')
for ri, inj in enumerate(injections):
    vals = mat[ri, ~np.isnan(mat[ri])]
    if len(vals):
        print(f'  {inj:45s}: {vals.mean():.3f}  (min={vals.min():.3f}, max={vals.max():.3f}, n={len(vals)})')

In [ ]:
# ── Per-injection feature importance ──────────────────────────────────────────
# For each injection, train a classifier on ALL its contexts (pooled 5-fold CV),
# extract |coef| → reshape to [L, H, G], plot layer×head heatmap.
# Allows comparison: which (layer, head) combinations are most important
# and do they overlap across injections?

X_feat  = FEAT_SETS[BEST_FEAT]
imp_by_inj = {}   # {inj_key: [L, H, G] importance array}

is_global = 'global' in BEST_FEAT
layers    = GLOBAL_LAYERS if is_global else list(range(N_LAYERS))
n_layers  = len(layers)

fig, axes = plt.subplots(len(INJECTIONS), 1,
                         figsize=(14, 4 * len(INJECTIONS)),
                         squeeze=False)

for ri, (suite, inj) in enumerate(INJECTIONS):
    inj_key  = f'{suite}/{inj}'
    mask     = inj_keys == inj_key
    X_inj    = X_feat[mask]
    y_inj    = y[mask]

    clf = make_clf()
    clf.fit(X_inj, y_inj)
    raw_imp = np.abs(clf.coef_[0])   # [n_features]

    # Reshape to [n_layers, H, G]
    imp_4d = raw_imp.reshape(n_layers, N_HEADS, N_GROUPS)
    imp_by_inj[inj_key] = imp_4d

    # Layer × head heatmap (sum over groups)
    ax = axes[ri, 0]
    lh = imp_4d.sum(-1)              # [n_layers, H]
    im = ax.imshow(lh.T, aspect='auto', cmap='viridis',
                   interpolation='nearest', origin='lower')
    ax.set_xlabel('Layer')
    ax.set_ylabel('Head')
    ax.set_title(f'{inj_key}  |coef| summed over span groups  ({BEST_FEAT})', fontsize=9)
    ax.set_xticks(range(n_layers))
    ax.set_xticklabels(
        [f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in layers], fontsize=6
    )
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.suptitle('Per-injection feature importance (|coef|, all contexts pooled)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top discriminative heads per injection ────────────────────────────────────
print(f'Top 10 (layer, head, group) by |coef| per injection ({BEST_FEAT})')
print()

for suite, inj in INJECTIONS:
    inj_key = f'{suite}/{inj}'
    imp_4d  = imp_by_inj[inj_key]    # [n_layers, H, G]
    flat    = imp_4d.reshape(-1)
    top_idx = np.argsort(flat)[::-1][:10]

    print(f'{inj_key}:')
    print(f'  {"Rank":>4}  {"Layer":>5}  {"L/G":>3}  {"Head":>4}  {"Group":>16}  {"| coef |":>9}')
    print('  ' + '-' * 50)
    for rank, idx in enumerate(top_idx, 1):
        li = idx // (N_HEADS * N_GROUPS)
        hd = (idx % (N_HEADS * N_GROUPS)) // N_GROUPS
        gi = idx % N_GROUPS
        lg = 'L' if layers[li] in LOCAL_LAYERS else 'G'
        print(f'  {rank:>4}  {layers[li]:>5}  {lg:>3}  {hd:>4}  '
              f'{GROUP_NAMES[gi]:>16}  {flat[idx]:>9.5f}')
    print()

In [ ]:
# ── Shared importance: heads that matter across ALL injections ────────────────
# Element-wise minimum of normalised importances — a head scores high here
# only if it is consistently important in every injection.

inj_keys_list = [f'{s}/{i}' for s, i in INJECTIONS]
if len(inj_keys_list) < 2:
    print('Need at least 2 injections for shared importance analysis.')
else:
    # Normalize each injection's importance to [0,1]
    norm_imps = []
    for ik in inj_keys_list:
        raw = imp_by_inj[ik].sum(-1)   # [n_layers, H]
        norm_imps.append(raw / (raw.max() + 1e-10))

    shared = np.stack(norm_imps, axis=0).min(axis=0)  # [n_layers, H]

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))

    # Shared minimum heatmap
    ax = axes[0]
    im = ax.imshow(shared.T, aspect='auto', cmap='hot',
                   interpolation='nearest', origin='lower')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_title('Shared importance (element-wise min of normalised |coef|)\n'
                 'High = consistently important across ALL injections', fontsize=9)
    ax.set_xticks(range(n_layers))
    ax.set_xticklabels(
        [f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in layers], fontsize=6
    )
    plt.colorbar(im, ax=ax, shrink=0.6)

    # Per-injection rank-correlation of importance (do injections agree on which heads matter?)
    ax = axes[1]
    flat_imps = np.stack([norm_imps[i].reshape(-1) for i in range(len(inj_keys_list))], axis=0)
    corr = np.corrcoef(flat_imps)
    im2 = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, interpolation='nearest')
    labels = [f'{s.split("/")[0]}\n{i.replace("injection_task_","it")}'
              for s, i in INJECTIONS]
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_title('Importance rank-correlation across injections\n'
                 'Do injections agree on which heads discriminate?', fontsize=9)
    for i in range(len(inj_keys_list)):
        for j in range(len(inj_keys_list)):
            ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=9)
    plt.colorbar(im2, ax=ax, shrink=0.6)

    plt.suptitle('Cross-injection importance agreement', fontsize=11)
    plt.tight_layout()
    plt.show()

    # Top shared (layer, head) pairs
    flat_shared = shared.reshape(-1)
    top_shared  = np.argsort(flat_shared)[::-1][:15]
    print('Top 15 (layer, head) by shared importance:')
    print(f'{"Rank":>4}  {"Layer":>5}  {"L/G":>3}  {"Head":>4}  {"shared_score":>12}')
    print('-' * 35)
    for rank, idx in enumerate(top_shared, 1):
        li = idx // N_HEADS
        hd = idx % N_HEADS
        lg = 'L' if layers[li] in LOCAL_LAYERS else 'G'
        print(f'{rank:>4}  {layers[li]:>5}  {lg:>3}  {hd:>4}  {flat_shared[idx]:>12.5f}')

In [ ]:
# ── Per-N AUROC per injection ─────────────────────────────────────────────────
# Signal at N=1 (single-token flip) confirms it's structural, not severity proxy.
X_feat   = FEAT_SETS[BEST_FEAT]
N_values = sorted(set(int(m.get('perturbation_N', 0)) for m in meta))

print(f'Per-N pooled CV AUROC per injection ({BEST_FEAT})')
print()

per_n_data = defaultdict(dict)  # {inj_key: {N: auroc}}

for suite, inj in INJECTIONS:
    inj_key  = f'{suite}/{inj}'
    inj_mask = inj_keys == inj_key
    print(f'{inj_key}:')

    for n_val in N_values:
        idx   = np.where(inj_mask & np.array(
            [int(m.get('perturbation_N', -1)) == n_val for m in meta]
        ))[0]
        if len(idx) < 10:
            continue
        X_n   = X_feat[idx]
        y_n   = y[idx]
        n_pos = int(y_n.sum())
        n_neg = int((1 - y_n).sum())
        if n_pos < 5 or n_neg < 5:
            print(f'  N={n_val:>3}: skipped (pos={n_pos}, neg={n_neg})')
            continue
        k     = min(5, n_neg)
        scores = cross_val_score(
            make_clf(), X_n, y_n,
            cv=StratifiedKFold(n_splits=k, shuffle=True, random_state=42),
            scoring='roc_auc', n_jobs=1,
        )
        per_n_data[inj_key][n_val] = scores.mean()
        note = ' ← single-token' if n_val == 1 else ''
        print(f'  N={n_val:>3}: AUROC={scores.mean():.4f} ± {scores.std():.4f}  '
              f'(n={len(idx)}, pos={n_pos}, neg={n_neg}){note}')
    print()

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.tab10(np.linspace(0, 0.8, len(INJECTIONS)))
for (suite, inj), color in zip(INJECTIONS, colors):
    inj_key = f'{suite}/{inj}'
    d       = per_n_data[inj_key]
    if d:
        ns   = sorted(d.keys())
        aucs = [d[n] for n in ns]
        label = f'{suite}/{inj.replace("injection_task_","it")}'
        ax.plot(ns, aucs, 'o-', color=color, label=label, linewidth=1.5, markersize=6)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, alpha=0.5, label='chance')
ax.set_xlabel('N (tokens flipped)')
ax.set_ylabel('Pooled CV AUROC')
ax.set_title(f'Per-N AUROC per injection ({BEST_FEAT})')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 70)
print('PER-INJECTION SUMMARY')
print('=' * 70)
print(f'Feature set: {BEST_FEAT}')
print()

for suite, inj in INJECTIONS:
    inj_key = f'{suite}/{inj}'
    loco    = loco_results[BEST_FEAT][inj_key]
    valid   = [v for v in loco.values() if v is not None]
    skipped = sum(1 for v in loco.values() if v is None)
    n1_auc  = per_n_data[inj_key].get(1)

    inj_mask = inj_keys == inj_key
    n_total  = inj_mask.sum()
    n_ctx    = len(set(ctx_keys[inj_mask]))

    print(f'{inj_key}')
    print(f'  n_runs={n_total}  n_contexts={n_ctx}  '
          f'success_rate={y[inj_mask].mean():.2f}')
    if valid:
        print(f'  LOCO AUROC: mean={np.mean(valid):.3f}  '
              f'min={min(valid):.3f}  max={max(valid):.3f}  '
              f'(skipped {skipped} imbalanced contexts)')
    else:
        print(f'  LOCO AUROC: no valid contexts (all skipped)')
    if n1_auc is not None:
        print(f'  N=1 AUROC: {n1_auc:.3f}  (confound check)')
    print()

print('Interpretation guide:')
print('  LOCO > 0.75: attention signature transfers to new user-task contexts')
print('  LOCO < 0.60: context-specific — classifier learns context shortcuts')
print('  N=1  > 0.65: genuine structural signal, not severity proxy')
print('  N=1  < 0.55: signal driven by perturbation magnitude, not structure')